# Notebook for LSTM

In [ ]:
#import all sorts of packages 
import numpy as np
import datetime as dt
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report


In [61]:
# load data and view
data = pd.read_csv("data/final_offshore_data_2017_2025.csv")
data.head()

#data.describe()

,year_mon_day,hour,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,full_datetime,capacity,volume,percentage,emission,emissionfactor,correct_days
0,20170101,1,213.586816,85.714286,86.428571,10206.75,95.714286,2017-01-01-01,873501,873501,1.014165,0,0,2017-01-01-01
1,20170101,2,210.905296,87.142857,90.714286,10199.75,96.142857,2017-01-01-02,883749,883749,1.026065,0,0,2017-01-01-02
2,20170101,3,208.585001,89.285714,87.857143,10191.50,96.000000,2017-01-01-03,872500,872500,1.013004,0,0,2017-01-01-03
3,20170101,4,209.977979,90.000000,90.000000,10182.25,96.142857,2017-01-01-04,889750,889750,1.033031,0,0,2017-01-01-04
4,20170101,5,208.541568,89.285714,87.142857,10176.25,95.571429,2017-01-01-05,893251,893251,1.037095,0,0,2017-01-01-05


In [62]:
#separate data into X and y
X = data.drop(["volume", "capacity",  #capacity & volume are the same
                    "emission", "emissionfactor"], #are always 0 and can be ignored
                      axis=1)

# our target variable
y = data["volume"]

## Pre-Processing

### Creating a Datetime-Variable for Date
Using the code from the SARIMAX-notebook, this section creates a date-variable in YYYY-MM-DD format, imposes the `datetime`as index and limits the data to before 2025. 

In [63]:
# Create a "date"-variable in YYYY-MM-DD format 
X["date"] = X['full_datetime'].str.rsplit('-', n=1).str[0]

X.head(1)

,year_mon_day,hour,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,full_datetime,percentage,correct_days,date
0,20170101,1,213.586816,85.714286,86.428571,10206.75,95.714286,2017-01-01-01,1.014165,2017-01-01-01,2017-01-01


In [64]:
# Convert 'correct days' to datetype format for further processing
X[['date', 'hour']] = X['correct_days'].str.rsplit('-', n=1, expand=True)

X['hour'] = X['hour'].astype(int) - 1

X['datetime'] = pd.to_datetime(X['date']) + pd.to_timedelta(X['hour'], unit='H')

/var/folders/xk/v_s45nd9057cbzm0wb_2259m0000gn/T/ipykernel_59440/988251242.py:6: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  X['datetime'] = pd.to_datetime(X['date']) + pd.to_timedelta(X['hour'], unit='H')


In [65]:
#removing data from 2025
X = X[X["date"] < "2025-01-01"]

# Set 'datetime' as index for the dataset
X.set_index('datetime', inplace=True)


X.head(3)

,year_mon_day,hour,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,full_datetime,percentage,correct_days,date
datetime,,,,,,,,,,,
2017-01-01 00:00:00,20170101,0,213.586816,85.714286,86.428571,10206.75,95.714286,2017-01-01-01,1.014165,2017-01-01-01,2017-01-01
2017-01-01 01:00:00,20170101,1,210.905296,87.142857,90.714286,10199.75,96.142857,2017-01-01-02,1.026065,2017-01-01-02,2017-01-01
2017-01-01 02:00:00,20170101,2,208.585001,89.285714,87.857143,10191.50,96.000000,2017-01-01-03,1.013004,2017-01-01-03,2017-01-01


### Train-Test-Split
Everything before 2024 will be used as training data, while the data for 2024 will serve as test data.

In [76]:
# Features
X_train = X[X["date"] < "2024-01-01"]
X_test = X[X["date"] >= "2024-01-01"]

# Label
y_train = y[:X_train.shape[0]] #list slice: from beginning to end of X-train
y_test = y[X_train.shape[0]:] #list slice: from end of X-train to the end of the data

X_train.head()

,year_mon_day,hour,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,full_datetime,percentage,correct_days,date
datetime,,,,,,,,,,,
2017-01-01 00:00:00,20170101,0,213.586816,85.714286,86.428571,10206.75,95.714286,2017-01-01-01,1.014165,2017-01-01-01,2017-01-01
2017-01-01 01:00:00,20170101,1,210.905296,87.142857,90.714286,10199.75,96.142857,2017-01-01-02,1.026065,2017-01-01-02,2017-01-01
2017-01-01 02:00:00,20170101,2,208.585001,89.285714,87.857143,10191.50,96.000000,2017-01-01-03,1.013004,2017-01-01-03,2017-01-01
2017-01-01 03:00:00,20170101,3,209.977979,90.000000,90.000000,10182.25,96.142857,2017-01-01-04,1.033031,2017-01-01-04,2017-01-01
2017-01-01 04:00:00,20170101,4,208.541568,89.285714,87.142857,10176.25,95.571429,2017-01-01-05,1.037095,2017-01-01-05,2017-01-01


### Scaling
For LSTM, the data needs to be scaled and normalization seems to be the way to go. However, we must be careful not to scale the temporal information.

In [ ]:
# subset dataframe to only contain the features we are interested in
numeric_features = ["wind_dir_avg_10", "wind_speed_h_avg", "wind_speed_avg_10", "air_pressure", "humidity", "percentage"]
X_train = X_train.loc[:, numeric_features]
X_test = X_test.loc[:, numeric_features]

# Define the preprocessor
preprocessor = ColumnTransformer([
    ("scaled_num", StandardScaler(), numeric_features)
])


# Transform the data with the scaler
X_train_norm = preprocessor.fit_transform(X_train)
X_test_norm = preprocessor.fit_transform(X_test)


array([[0.23713527, 0.6755876 , 0.69600044, 0.53874442, 1.29614683,
        1.56232934],
       [0.20935343, 0.72523504, 0.84464269, 0.47421159, 1.33424613,
        1.59611529],
       [0.18531406, 0.7997062 , 0.74554786, 0.39815503, 1.32154637,
        1.55903279],
       [0.19974598, 0.82452992, 0.81986898, 0.31287951, 1.33424613,
        1.61589291],
       [0.18486407, 0.7997062 , 0.72077415, 0.25756565, 1.28344707,
        1.62743119]])

In [78]:
#add feature names back 
X_train = pd.DataFrame(X_train_norm, columns=X_train.columns.values, index=X_train.index.values)
# X_train.head()

X_test = pd.DataFrame(X_test_norm, columns = X_test.columns.values, index=X_test.index.values)

#X_train = X_train.loc[:, numeric_features]
X_train.head()

,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,percentage
2017-01-01 00:00:00,0.237135,0.675588,0.696000,0.538744,1.296147,1.562329
2017-01-01 01:00:00,0.209353,0.725235,0.844643,0.474212,1.334246,1.596115
2017-01-01 02:00:00,0.185314,0.799706,0.745548,0.398155,1.321546,1.559033
2017-01-01 03:00:00,0.199746,0.824530,0.819869,0.312880,1.334246,1.615893
2017-01-01 04:00:00,0.184864,0.799706,0.720774,0.257566,1.283447,1.627431


In [ ]:
#let us verify the z-score standardization manually
wind = data.loc[:, "wind_dir_avg_10"]

wind_normalized = (wind - wind.mean()) / wind.std()
wind_normalized

0        0.234679
1        0.206700
2        0.182490
3        0.197024
4        0.182037
           ...   
70146    0.568435
70147    0.494891
70148    0.585283
70149    0.701167
70150    1.120796
Name: wind_dir_avg_10, Length: 70151, dtype: float64

### Create 3D-Dataset
For LSTM to work, the data needs to remodelled to be in 3D shape (see Brownlee, 2018). Following Brownlee, LSTMs work best if each sample contains between 200-400 time steps (page 48), where of course each sample contains the same amount of time steps. Hence, we need to split the data into evenly sized samples first. In principle, this is something we can play around with at a later stage: if performance is not good, we could iterate over the indicated sample size range and see if that way improvements can be achieved. 

In [99]:
# function for creating 3D datasets
def create_3d_data(X_train, X_test, y_train, y_test):
    # define amount of time steps per sample 
    sample_steps_X_train = range(0, len(X_train), #can be recycled for y_train
                         400) # step size is 400 because literature suggests 200-400. We can play with this to perhaps reach improvements
    sample_steps_X_test = range(0, len(X_test), 400) # can be recycled for y_test 


    #initialize empty lists to store results 
    train_data_X, test_data_X, train_data_y, test_data_y = [], [], [], []

    # samples for X data
    for i, step in enumerate(sample_steps_X_train[:-1]): #sample steps only enumerates until penultimate element because otherwise i+1 creates an out of range error in the last loop iteration
       #ensure we stay within limits of data set 
        if sample_steps_X_train[i+1] > len(X_train):
           break

        #train data 
        train_data_X.append(X_train[step:sample_steps_X_train[i+1]])

        #test data 
    for i, step in enumerate(sample_steps_X_test[:-1]):
        #ensure we stay within limits of data set 
        if sample_steps_X_test[i+1] > len(X_test):
           break
        test_data_X.append(X_test[step:sample_steps_X_test[i+1]])

    #samples for y data 
    interval = 7*24 # we want to predict an interval of 1 week = 7*24h
    for i, step in enumerate(sample_steps_X_train[1:-1]): #we start at the end of each train_data sample
        #ensure we stay within limits of data set 
        #if step + interval > len(y_train):
         # break
        train_data_y.append(y_train[step:step+interval]) #we go from the end of the train_data sample to the end of the interval we want to predict

    for i, step in enumerate(sample_steps_X_test[1:-1]):
        if step + interval > len(y_test):
            break
        test_data_y.append(y_test[step:step+interval])
    


    return np.array(train_data_X), np.array(test_data_X), np.array(train_data_y), np.array(test_data_y)



X_train_3d, X_test_3d, y_train_3d, y_test_3d = create_3d_data(X_train, X_test, y_train, y_test)

# we have to do this because otherwise the y-datasets are one sample short 
# arrays need to be transformed to tensors to fit the model
X_train_3d = torch.Tensor(X_train_3d[:-1])
X_test_3d = torch.Tensor(X_test_3d[:-1])
y_train_3d = torch.Tensor(y_train_3d)
y_test_3d = torch.Tensor(y_test_3d)

### Modelling
Create a crude model

In [110]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, batch_first):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first = batch_first)
        self.fc = nn.Linear(hidden_dim, output_dim)

    #hidden layer 
    def forward(self, x, h0=None, c0=None):
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out, hn, cn





In [130]:
#initiate a model 
model = LSTMModel(input_dim=6, hidden_dim=24, layer_dim=2, output_dim = 168, batch_first=True)

# loss function 
criterion = nn.CrossEntropyLoss()
# defines learning rate and gradient method 
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# iterate over epochs
epochs = 100
hidden, cn = None, None # these network layers are initially not defined 

for i in range(epochs):
    model.train()
    optimizer.zero_grad()
    output, hidden, cn = model(X_train_3d, hidden, cn)
    loss = criterion(output, y_train_3d)
    loss.backward()
    optimizer.step()

    hidden = hidden.detach()
    cn = cn.detach()

    if (i+1) % 10 == 0:
        print(f"Epoch [{i+1}/{epochs}], Loss: {loss.item():.4f}")






Epoch [10/100], Loss: 604841216.0000
Epoch [20/100], Loss: 602982720.0000
Epoch [30/100], Loss: 600593472.0000
Epoch [40/100], Loss: 597959168.0000
Epoch [50/100], Loss: 596249600.0000
Epoch [60/100], Loss: 595074304.0000
Epoch [70/100], Loss: 593461056.0000
Epoch [80/100], Loss: 592090432.0000
Epoch [90/100], Loss: 591053952.0000
Epoch [100/100], Loss: 589929536.0000


In [129]:
predictions

tensor([[-1.4722e+00, -1.5600e+00, -1.6632e+00,  ..., -1.3438e-01,
         -7.0150e-02, -6.4595e-02],
        [ 4.3687e-01,  4.5460e-01,  5.4721e-01,  ...,  5.3039e-01,
          4.5485e-01,  4.1204e-01],
        [ 3.9587e-02,  1.1837e-02, -1.2659e-03,  ...,  2.7998e-01,
          2.9881e-01,  2.5859e-01],
        ...,
        [-3.2873e-01, -2.4281e-01, -4.1848e-01,  ...,  1.5584e+00,
          1.6143e+00,  1.6736e+00],
        [ 2.4452e-01,  9.0092e-02,  4.5487e-03,  ...,  1.1578e+00,
          1.3347e+00,  1.3531e+00],
        [-2.0805e+00, -1.9812e+00, -2.0048e+00,  ...,  6.0107e-01,
          5.7792e-01,  5.5763e-01]])

In [128]:
# evaluation of our model
model.eval()
with torch.no_grad(): #no_grad = ensure the model does not keep learning
    predictions, hidden, cn = model(X_test_3d)
    lstm_eval = accuracy_score(y_test_3d, predictions)

lstm_eval

ValueError: Classification metrics can't handle a mix of multiclass-multioutput and continuous-multioutput targets